<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/LoRA_QLoRA_Local_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning Open Models Locally with LoRA and QLoRA

Earlier in this section you fine-tuned a model you never actually held: you uploaded a JSONL file to a provider, waited, and got back a model ID. This lesson does the opposite. The weights land on **your** GPU, the training loop runs in **this** process, and what you get back is a folder of files you own — no vendor, no per-token bill, no rules about what you may train on.

The reason that is possible on a free Colab GPU is two ideas stacked. **LoRA** freezes the pretrained weights and trains a small low-rank update beside them, so you optimize a tiny fraction of the parameters. **QLoRA** adds one more move: squeeze those frozen weights down to 4 bits. Together they let a 20-billion-parameter open reasoning model — `gpt-oss-20b` — train on a card that has no business hosting it.

📎 *The theory behind both was introduced in **Fine-Tuning 101**. This notebook is where it stops being theory and becomes a training run you can watch.*

## 🧭 What You'll Learn

- Why LoRA works at all — the **low-rank decomposition** behind it, and the arithmetic that turns a `d × d` update into two thin matrices
- What the **"Q"** adds: a 4-bit frozen base, higher-precision adapters, and the quantization memory ladder that makes a free T4 enough
- The full local recipe: load a 4-bit base, attach adapters, format a chat dataset, and train with TRL's `SFTTrainer`
- **Training on completions only** — masking the prompt out of the loss, why it matters, and how to prove the mask actually landed
- Saving adapters, merging them for serving, and the judgement call of when to fine-tune at all instead of prompting or retrieving

## 1. Setup: A GPU Runtime, and No API Keys This Time

**This notebook does not run on CPU.** Not slowly — at all. The 4-bit quantization and the fused training kernels used here are CUDA code, so without an NVIDIA GPU the model-loading cell fails outright. That is the one hard requirement of the lesson, and it is worth checking *before* you install several gigabytes of wheels.

The good news is that the requirement is modest, and that is the whole point of the lesson: a **free Colab T4 is enough for a 20B model**, precisely because the frozen base is stored in 4 bits. To switch the runtime in Colab: **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**, then rerun the notebook from the top. Locally, a recent NVIDIA card with T4-class memory (16 GB) or more plays the same role; Apple Silicon and CPU-only machines do not.

**Environment note, in place of the usual pickers:** every other notebook in this section opens with `PROVIDER` / `CHAT_MODEL` dropdowns because it calls a hosted chat API. This one never does — the model runs on the GPU attached to this runtime — so there is no provider to choose and **no API key to set**. The only account you might need is a Hugging Face one for gated weights, and the model used here is not gated.

In [1]:
# A ten-second check that saves a ten-minute install: is there a CUDA GPU here?
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    print("❌ No NVIDIA GPU visible in this runtime.")
    print("   Colab: Runtime → Change runtime type → T4 GPU → Save, then rerun.")
    print("   Local: this lesson needs an NVIDIA card — CPU and Apple Silicon cannot run it.")
else:
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
        check=True,
    )

With a GPU confirmed, one setup cell installs the pinned profile. It is a heavier install than the rest of the course — a *training* stack, not just an SDK — so expect a few minutes on a fresh runtime.

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
import importlib
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

# Pinned course profile for this lesson (July 2026). Unlike every other notebook
# in this section, this is a TRAINING stack: the model runner (unsloth), the
# transformer library it patches, the trainer (trl), the adapter implementation
# (peft), and the 4-bit quantizer (bitsandbytes).
PINS = [
    "unsloth",
    "unsloth_zoo",
    "transformers", # Unpinned to allow for compatible versions
    "trl",          # Already unpinned in previous step
    "peft",         # Unpinned to allow for compatible versions
    "bitsandbytes", # Unpinned to allow for compatible versions
    "datasets",
    "accelerate",
]

if IN_COLAB:
    import site

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PINS], check=True)
    importlib.reload(site)  # make the new packages importable without a runtime restart
else:
    # Locally: install the same pins once into your project environment
    # (pip install " ".join(PINS)), then just run the notebook.
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

import torch

assert torch.cuda.is_available(), (
    "No CUDA GPU available — see the runtime note above. This notebook cannot run on CPU."
)

print(f"✅ Setup complete — {'Colab' if IN_COLAB else 'local'} | GPU: {torch.cuda.get_device_name(0)}")

✅ Setup complete — Colab | GPU: Tesla T4


## 2. What LoRA Actually Does

Full fine-tuning updates every weight in the model. For a 20B model that means holding 20 billion parameters, **plus** a gradient for each, **plus** optimizer state for each — which is why full fine-tuning of anything interesting needs a cluster rather than a Colab tab.

LoRA's observation is that the *update* you want is far simpler than the model you are updating. Take one weight matrix `W`, shape `d × d`. Fine-tuning wants to replace it with `W + ΔW`. LoRA never forms `ΔW` at full size — it writes it as a product of two thin matrices:

```
ΔW  ≈  B @ A        A is r × d,  B is d × r,  with r ≪ d
```

`W` stays **frozen**: untouched, still carrying everything pretraining put there. Only `A` and `B` receive gradients. The forward pass becomes `h = W @ x + (lora_alpha / r) · (B @ A) @ x` — the original model, plus a small learned correction sitting alongside it.

The arithmetic is the entire pitch. A `d × d` update has `d²` parameters; its rank-`r` stand-in has `2 · d · r`. The ratio simplifies to `2r / d`, which is a number worth computing rather than taking on faith.

In [4]:
def lora_param_ratio(d, r):
    """Parameters in a full d×d update vs its rank-r LoRA stand-in (B @ A)."""
    full = d * d
    lora = 2 * d * r  # A is r×d, B is d×r
    return full, lora, lora / full


print(f"{'d':>6} {'r':>4} {'full update':>14} {'LoRA (B@A)':>12} {'share':>8}")
for d in (1024, 2048, 4096):
    for r in (8, 16, 64):
        full, lora, share = lora_param_ratio(d, r)
        print(f"{d:>6} {r:>4} {full:>14,} {lora:>12,} {share:>7.2%}")

     d    r    full update   LoRA (B@A)    share
  1024    8      1,048,576       16,384   1.56%
  1024   16      1,048,576       32,768   3.12%
  1024   64      1,048,576      131,072  12.50%
  2048    8      4,194,304       32,768   0.78%
  2048   16      4,194,304       65,536   1.56%
  2048   64      4,194,304      262,144   6.25%
  4096    8     16,777,216       65,536   0.39%
  4096   16     16,777,216      131,072   0.78%
  4096   64     16,777,216      524,288   3.12%


**What just happened?** You computed `2r / d` for a few realistic shapes. Read the `share` column as the answer to "how much of this matrix am I actually training?" — and notice the two directions it moves: the wider the matrix, the *cheaper* LoRA gets relative to it, and the higher the rank, the more capacity you buy back. `r` is the dial between "too small to learn the task" and "large enough to memorize the dataset".

Two consequences are worth carrying forward. First, the frozen base **cannot** suffer catastrophic forgetting, because nothing overwrote it — the worst a bad adapter can do is speak too loudly, and you can turn it off by not loading it. Second, the artefact you train is a few megabytes of adapter rather than a copy of the model, so a dozen tasks can share one base model and swap adapters at load time.


## 3. The "Q" in QLoRA: Why This Fits on a Free GPU

LoRA shrinks what you **train**. It does nothing about what you **store** — the frozen base still has to sit in VRAM to run the forward pass, and for a 20B model at 16 bits that is the whole problem again.

QLoRA's addition is one sentence: **quantize the frozen base to 4 bits, and keep the adapters at higher precision.** The base weights are stored in NF4, a 4-bit data type shaped for normally-distributed weights, dequantized on the fly for each matmul, and never updated — so their precision loss never compounds across steps. Gradients flow *through* that frozen 4-bit base and land only in `A` and `B`, which stay in 16-bit, where small updates actually need the resolution.

The reason this is the enabling trick rather than a nice optimization is the memory ladder — bytes per parameter, by precision.

In [5]:
# Weight memory ONLY — this is the floor, not the whole bill. Activations,
# gradients, optimizer state and the KV cache all sit on top of these numbers.
BYTES_PER_PARAM = {"fp32": 4, "fp16 / bf16": 2, "int8": 1, "4-bit (nf4)": 0.5}

for params_b in (8, 20):
    print(f"\n{params_b}B parameters — weights alone:")
    for precision, nbytes in BYTES_PER_PARAM.items():
        print(f"  {precision:>12}: {params_b * 1e9 * nbytes / 1024 ** 3:6.1f} GiB")


8B parameters — weights alone:
          fp32:   29.8 GiB
   fp16 / bf16:   14.9 GiB
          int8:    7.5 GiB
   4-bit (nf4):    3.7 GiB

20B parameters — weights alone:
          fp32:   74.5 GiB
   fp16 / bf16:   37.3 GiB
          int8:   18.6 GiB
   4-bit (nf4):    9.3 GiB


Those are *weight* figures derived from first principles — parameter count times bytes per parameter — and they are a floor, not a forecast. A real training run also holds activations, gradients, optimizer state and a KV cache on top. Read the ladder as the **shape** of the argument (each rung halves the floor), not as a VRAM prediction for your run.


**What just happened?** You separated two claims that usually get blurred. Quantization is what makes the model *fit*; LoRA is what makes training it *affordable*. Neither alone gets a 20B model onto a T4 — 4-bit weights with full fine-tuning still needs gradients and optimizer state for 20 billion parameters, and LoRA on 16-bit weights still needs the ~40 GB of weights the ladder above prices (20B × 2 bytes) before training even starts. QLoRA is the product of the two, and that product is what this notebook runs.

## 4. Load the Base Model in 4-Bit

The subject is **gpt-oss-20b**, OpenAI's open-weight reasoning model. Two properties make it a good one: it is genuinely large, so the memory argument above is not hypothetical, and it exposes a `reasoning_effort` control that lets you trade thinking tokens for latency at inference time.

`FastLanguageModel.from_pretrained` is Unsloth's loader. It fetches the weights, applies patched attention and MLP kernels, and — with `load_in_4bit=True` — returns a model whose frozen base already lives in 4 bits. `max_seq_length` is a memory decision as much as a capability one: attention cost grows with sequence length, and 1024 tokens keeps a T4 comfortable.

In [6]:
# @title ⚙️ Base model and context length { display-mode: "form" }
BASE_MODEL = "unsloth/gpt-oss-20b"  # @param ["unsloth/gpt-oss-20b", "unsloth/gpt-oss-20b-unsloth-bnb-4bit", "unsloth/gpt-oss-120b"] {allow-input: true}
MAX_SEQ_LENGTH = 1024  # @param {type:"integer"}
# gpt-oss-20b ships in MXFP4 and is loaded here in 4-bit; the -unsloth-bnb-4bit
# repo is the same model pre-quantized with bitsandbytes (smaller download, no
# quantization step at load). The 120B variant will NOT fit a free T4.

In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,             # None → pick the best dtype for this GPU automatically
    load_in_4bit=True,      # the "Q" in QLoRA: the frozen base is stored in 4 bits
    full_finetuning=False,  # we train adapters, not all 20B parameters
    # token="hf_...",       # only needed for gated repos
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.19: Fast Gpt_Oss patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


Loading weights:   0%|          | 0/3387 [00:00<?, ?it/s]

Before changing anything, meet the model. `gpt-oss` accepts a `reasoning_effort` argument in its chat template — `"low"`, `"medium"` or `"high"` — controlling how many tokens it spends thinking before it answers. It is a real cost dial: higher effort generally means better multi-step answers and a longer wait, and it is the same trade-off the reasoning-model lesson framed in the abstract.

One practical detail: raise `max_new_tokens` when you raise the effort, or the answer gets cut off mid-thought and looks like a model failure when it is a budget failure.

In [8]:
from transformers import TextStreamer

QUESTION = [{"role": "user", "content": "Solve x⁴ - 6x³ + 11x² - 6x = 0"}]


def ask(messages, reasoning_effort="low", max_new_tokens=256):
    """Generate one reply on the GPU, streaming tokens as they arrive."""
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        reasoning_effort=reasoning_effort,  # "low" | "medium" | "high"
    ).to("cuda")
    return model.generate(
        **inputs, max_new_tokens=max_new_tokens, streamer=TextStreamer(tokenizer)
    )


_ = ask(QUESTION, reasoning_effort="low", max_new_tokens=128)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-08-21

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve x⁴ - 6x³ + 11x² - 6x = 

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0<|end|><|start|>assistant<|channel|>analysis<|message|>We need roots: factor: x(x^3 -6x^2+11x-6)=0. The cubic is (x-1)(x^2-? Wait it's (x-1)(x^2-? 6?). Let's factor: try x=1 gives 1-6= -? Actually 11x? Wait. Let's compute polynomial: x^3 - 6x^2 + 11x - 6 = x*(x^2 -6x+11)?? Wait.

But we can try to factor as (x-1)(x-?).

Try


Same question, more thinking budget — and enough room to spend it.

In [9]:
_ = ask(QUESTION, reasoning_effort="high", max_new_tokens=512)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-08-21

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve x⁴ - 6x³ + 11x² - 6x = 

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0<|end|><|start|>assistant<|channel|>analysis<|message|>We have to solve the equation: x^4 - 6x^3 + 11x^2 - 6x = 0. This is a quartic. We could factor. Let's first factor out x? Let me check: x^4 - 6x^3 + 11x^2 - 6x = 0.

However, the polynomial is missing the constant term; we can factor out x: x^4 - 6x^3 + ... = x(x^3 - 6x^2 + 11x - something?). Wait. Actually x^4 - 6x^3 + 11x^2 - 6x = x^4 - 6x^3 + 6x^2 - 0? Wait I think it should be factorized. Let's attempt to factor as polynomial in x. We see that the polynomial is similar to the expression (x^2 - 3x + 1)(x^2 - 1), maybe something like (x - 1)(x^3 - 5x^2 + ...)? Wait. Let's check.

We can attempt to factor it by grouping terms of degree 3 and 2, but maybe it's a standard polynomial of degree 3 with missing linear term.

Let's start: x^4 - 6x^3 + 11x^2 - 6x = 0
That's a quartic in form. We can try to factor it by grouping as (x^2 - 3x)(something). Actually, maybe we can factor as (x^2 - 1)(x^2 - 5x...). Wait: we can factor further.

**What just happened?** A 20-billion-parameter model loaded onto a free GPU and answered, with its weights quantized to 4 bits and not one of them trainable yet. That is the baseline: everything from here changes *behaviour*, and you now have a before-picture to compare against.

Notice also that `reasoning_effort` is a knob you get for free at inference time, independent of anything you fine-tune. If your quality problem is "the model rushes," try the cheap dial before reaching for a training run.

## 5. Attach the LoRA Adapters

`get_peft_model` walks the loaded model, finds the matrices you name, and inserts an `A`/`B` pair beside each one. Every argument is a decision, so here is what each one decides:

- **`r`** — the rank from Section 2, and the capacity dial. Higher `r` means more trainable parameters and more room to learn; too high on a small dataset just memorizes it. 8–64 is the usual working range.
- **`lora_alpha`** — the scaling factor. The adapter enters the forward pass as `(lora_alpha / r) · B @ A`, so alpha sets **how loudly** it speaks relative to the frozen base. Keeping `lora_alpha == r` (scale 1.0) is a sane, boring default; `2 × r` is the other common convention.
- **`target_modules`** — which matrices get adapters. The list below covers the attention projections (`q/k/v/o_proj`) **and** the MLP (`gate/up/down_proj`). Attention-only is cheaper; adapting both is the modern default and usually worth it.
- **`lora_dropout`** — regularization on the adapter path. `0.0` takes Unsloth's optimized kernel; raise it if a small dataset starts overfitting.
- **`bias="none"`** — do not train bias terms. Fewer parameters, no measurable loss in practice.
- **`use_gradient_checkpointing`** — recompute activations during the backward pass instead of storing them: slower per step, much cheaper in memory. `"unsloth"` picks the library's own implementation, tuned for long context. This is often the difference between fitting and OOM.
- **`random_state`** — the seed. `A` and `B` are randomly initialized, so a fixed seed is what makes two runs comparable at all.

**A word on the numbers below — they are starting points, not truths.** `r=16, lora_alpha=16`, with `learning_rate=1e-4` over `2` epochs in the next section, is a conservative default for a real fine-tune on your own data. The recipe this lesson is built from used `r=8`, `lora_alpha=16` and `lr=2e-4` for a **30-step demo run** — a configuration designed to finish in minutes and show the mechanics, where a higher learning rate is exactly right because there are so few steps in which to learn anything. Neither set is wrong; they answer different questions. Keep the demo values if you want to watch the loop turn, and the values below if you are actually trying to teach the model something.

In [10]:
# @title ⚙️ LoRA knobs { display-mode: "form" }
LORA_R = 8  # @param [8, 16, 32, 64] {type:"raw", allow-input: true}
LORA_ALPHA = 16  # @param [8, 16, 32, 64] {type:"raw", allow-input: true}
LORA_DROPOUT = 0.0  # @param {type:"number"}
SEED = 3407  # @param {type:"integer"}

In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,                   # rank — the capacity of the update (Section 2)
    lora_alpha=LORA_ALPHA,      # scale — adapter enters as (alpha / r) · B @ A
    lora_dropout=LORA_DROPOUT,  # 0.0 uses the optimized kernel
    bias="none",                # do not train bias terms
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention projections
        "gate_proj", "up_proj", "down_proj",     # MLP projections
    ],
    use_gradient_checkpointing="unsloth",  # trade compute for memory
    random_state=SEED,                     # reproducible adapter initialization
    use_rslora=False,   # rank-stabilized LoRA: rescales alpha by sqrt(r) instead of r
    loftq_config=None,  # LoftQ-style adapter init — off
)

model.print_trainable_parameters()  # trainable vs total — the LoRA pitch, measured

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Unsloth: Detected MoE model with per-expert Linear experts. Enabling LoRA on 64 expert projection modules.
trainable params: 92,454,912 || all params: 21,007,212,096 || trainable%: 0.4401


**What just happened?** The printed line is Section 2's arithmetic applied to every adapted matrix at once, with the denominator being the *entire* 20B model. Whatever percentage you see, read it as "this is the slice of the network that will move during training" — everything else is a frozen, 4-bit constant.

Two things changed silently and are worth knowing about. The model object is now a `PeftModel` wrapping the original, so saving it saves **adapters**, not weights. And gradient checkpointing is on, which means the backward pass recomputes activations instead of reading them — the step is slower, and the memory it frees is what buys you the batch size.

## 6. A Chat-Formatted Dataset

Supervised fine-tuning needs examples of the behaviour you want: a conversation ending in the assistant reply you wish the model had written. The format is the same `[{"role": ..., "content": ...}]` list the chat APIs have taken in every lesson since the first one — which is the useful part, because a dataset you assembled for evaluation can usually become a training set without reshaping.

This lesson borrows `HuggingFaceH4/Multilingual-Thinking`: reasoning chain-of-thought examples whose questions were translated into four non-English languages. It is small, it is public, and the behaviour it teaches is easy to **see** afterwards — the tuned model reasons in the target language, which is hard to fake and hard to miss. That matters for a teaching notebook: a fine-tune whose effect you cannot observe teaches you nothing about whether it worked.

Two steps turn it into training text. `standardize_data_formats` normalizes column names (`from`/`value`, `human`/`gpt` and friends all become `role`/`content`), and the tokenizer's own chat template then renders each conversation into the exact string the model was pretrained to expect — special tokens, channel markers and all. **Applying the model's own template is not a formality.** Train on a format the model does not use and you spend the entire run teaching it to unlearn its own prompt structure.

In [12]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_data_formats

dataset = load_dataset("HuggingFaceH4/Multilingual-Thinking", split="train")
dataset = standardize_data_formats(dataset)  # older code calls this standardize_sharegpt


def formatting_prompts_func(examples):
    """Render each conversation with the model's OWN chat template."""
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["messages"]
    ]
    return {"text": texts}


dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset)

README.md:   0%|          | 0.00/3.06k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 5.29MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['reasoning_language', 'developer', 'user', 'analysis', 'final', 'messages', 'text'],
    num_rows: 1000
})


Always look at one rendered example before training on ten thousand of them.

In [13]:
print(dataset[0]["text"])

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-08-21

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

reasoning language: French

You are an AI chatbot with a lively and energetic personality.<|end|><|start|>user<|message|>Can you show me the latest trends on Twitter right now?<|end|><|start|>assistant<|channel|>analysis<|message|>D'accord, l'utilisateur demande les tendances Twitter les plus récentes. Tout d'abord, je dois vérifier si j'ai accès à des données en temps réel. Étant donné que je ne peux pas naviguer sur Internet ou accéder directement à l'API de Twitter, je ne peux pas fournir des tendances en direct. Cependant, je peux donner quelques conseils généraux sur la façon de les trouver.

Je devrais préciser que les 

**What just happened?** The printed string is *literally* what the model will be trained on — not a summary of it. gpt-oss uses OpenAI's **Harmony** format, so you will see channel markers separating the model's reasoning from its final answer, plus the role tokens that open each turn. Those exact markers come back in Section 8, when we tell the trainer which part of this string it is allowed to learn from.

📎 *To train on your own data instead, produce a dataset with a `messages` column of role/content dicts — the same shape as the question/answer pairs you have been generating since the evaluation lesson — and the two cells above are unchanged.*

**And the rule that decides whether any of this works: quality beats quantity.** A few hundred examples that are genuinely the behaviour you want will beat tens of thousands of mediocre ones, because fine-tuning has no mechanism for ignoring a bad example — every one of them is a lesson the model dutifully learns. Before scaling a dataset up, read a random sample of it and ask whether you would be happy for the model to imitate each one exactly.

## 7. Configure the Trainer

TRL's `SFTTrainer` is a thin, well-behaved wrapper over the Hugging Face `Trainer`. It takes the model (already carrying its adapters), the dataset (already carrying its `text` column), and a config object where the decisions live:

- **`per_device_train_batch_size` × `gradient_accumulation_steps`** is your *effective* batch size. On a small GPU the first number is memory-bound and stays tiny, so you buy batch size with the second one — accumulate gradients over several forward passes and step once. Same math, less VRAM, more wall-clock.
- **`warmup_steps`** ramps the learning rate up from zero so the first few steps do not shock freshly-initialized adapters.
- **`max_steps` vs `num_train_epochs`** is the demo/real switch. `max_steps` stops after a fixed number of optimizer steps regardless of dataset size; epochs make full passes. `-1` disables the step cap.
- **`learning_rate`** is higher than you would dare on a full fine-tune. You are training a small number of fresh parameters, not nudging pretrained ones, so LoRA tolerates — and needs — a larger LR.
- **`optim="adamw_8bit"`** keeps optimizer state in 8 bits. Adam holds two state tensors per trainable parameter; with LoRA that set is already small, and 8-bit shrinks it again.
- **`logging_steps=1`** prints the loss every step. This is your instrument panel — on a short run it is the only thing telling you whether anything is happening.
- **`report_to="none"`** keeps the run local; swap in a tracker for a real one.

In [14]:
# @title ⚙️ Training knobs { display-mode: "form" }
DEMO_RUN = True  # @param {type:"boolean"}
MAX_STEPS = 30  # @param {type:"integer"}
NUM_EPOCHS = 2  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 1  # @param {type:"integer"}
GRAD_ACCUM = 4  # @param {type:"integer"}
# DEMO_RUN=True stops after MAX_STEPS so you can watch the loop turn in minutes —
# that is a mechanics demo, not a fine-tune. Set it to False for the real thing:
# NUM_EPOCHS full passes over the data. See the note in Section 5 on why the
# demo recipe also runs a higher learning rate than the default below.

In [15]:
from trl import SFTConfig, SFTTrainer

config = SFTConfig(
    per_device_train_batch_size=BATCH_SIZE,   # sequences per step per GPU — memory-bound
    gradient_accumulation_steps=GRAD_ACCUM,   # effective batch = BATCH_SIZE × GRAD_ACCUM
    warmup_steps=5,                           # ramp the LR from 0 so early steps don't shock the adapters
    max_steps=MAX_STEPS if DEMO_RUN else -1,  # -1 → ignore the step cap, run NUM_EPOCHS instead
    num_train_epochs=NUM_EPOCHS,              # full passes over the dataset (real runs)
    learning_rate=LEARNING_RATE,              # LoRA tolerates a higher LR than full fine-tuning
    max_length=MAX_SEQ_LENGTH,                # must not exceed what the model was loaded with
    logging_steps=1,                          # print the loss every step — your instrument panel
    optim="adamw_8bit",                       # 8-bit optimizer states: the other big memory saving
    weight_decay=0.001,                       # mild regularization
    lr_scheduler_type="linear",               # decay toward 0 by the final step
    seed=SEED,                                # same seed as the adapter init
    output_dir="outputs",
    report_to="none",                         # swap for a tracker on a real run
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,  # TRL 1.x name for what used to be `tokenizer=`
    args=config,
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

## 8. Train on Completions Only — and Run It

By default the trainer computes loss over **every** token in the rendered example, including the user's question. Think about what that actually teaches: the model is graded on its ability to predict the prompt — a skill nobody wants, paid for out of the gradient budget you meant to spend on the answer.

It is worse than merely wasteful. On a dataset with a repetitive prompt shape, predicting the prompt is *easy* signal: the loss curve falls, the run looks healthy, and what the model has learned is to parrot your instruction format rather than answer it. A masked run and an unmasked run can produce similar-looking loss charts and very different models.

The fix is masking: set the label for every prompt token to `-100`, the sentinel that means "ignore me in the loss". `train_on_responses_only` does exactly that, and it needs to know where the prompt ends and the answer begins — hence the two template markers below, which are specific to gpt-oss's Harmony format. Change the model family, change the markers.

In [16]:
from unsloth.chat_templates import train_on_responses_only

# The exact strings that open the user turn and the assistant's FINAL answer in
# gpt-oss's Harmony chat template (you saw both in the rendered example above).
# A different model family uses different markers — copy them from its template.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start|>user<|message|>",
    response_part="<|start|>assistant<|channel|>final<|message|>",
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Removed 67 out of 1000 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


Never take a masking step on faith — it fails **silently**. If the markers do not match the template, nothing raises; you just quietly train on everything. So print one example twice: once as the model sees it (`input_ids`), once as the loss sees it (`labels`, with ignored positions swapped back to a visible pad token).

In [17]:
example = trainer.train_dataset[100]

print("── what the MODEL sees (input_ids) " + "─" * 34)
print(tokenizer.decode(example["input_ids"]))

── what the MODEL sees (input_ids) ──────────────────────────────────
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-08-21

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

reasoning language: Spanish

You are an AI that guides users through the process of preparing a pasta recipe<|end|><|start|>user<|message|>I need to book a wheelchair accessible taxi for a trip to the airport tomorrow morning at 8am. Can you help with this?<|end|><|start|>assistant<|channel|>analysis<|message|>Perfecto, el usuario está pidiendo ayuda para reservar un taxi accesible para una persona en silla de ruedas para un viaje al aeropuerto mañana a las 8 de la mañana. Empezaré por comprender su solicitud. Necesitan un taxi accesible, por lo que es impo

Now the same row as the **loss** sees it. Every position the mask turned into `-100` is swapped back to the pad token so it becomes visible whitespace — what remains is exactly what the model is being graded on.

In [18]:
print("── what the LOSS sees (labels; -100 → padding) " + "─" * 22)
print(
    tokenizer.decode(
        [tokenizer.pad_token_id if t == -100 else t for t in example["labels"]]
    ).replace(tokenizer.pad_token, " ")
)

── what the LOSS sees (labels; -100 → padding) ──────────────────────
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                Certainly! Here's a step-by-step guide to help you book a wheelchair-accessible taxi for your airport trip tomorrow at 8:00 AM:

---

### **1. Confirm Your Location**
I’ll need to know your **city or airport name** to provide accurate options. Different regions have different services. Let

The second print should be almost entirely whitespace, with only the assistant's answer surviving. If you can still read the user's question in it, the markers do not match your model's template and the masking did nothing at all.

*(TRL 1.x ships its own equivalent — `assistant_only_loss=True` in `SFTConfig` — which derives the boundaries from the chat template instead of hand-written strings. Either approach is fine; do not enable both.)*

Now train. Watch the loss column as it goes: it should fall unevenly. A curve that is flat from step one usually means every label got masked; a curve that plunges to near-zero on a small dataset usually means memorization rather than learning.

In [19]:
# @title 📊 GPU memory before training { display-mode: "form" }
gpu = torch.cuda.get_device_properties(0)
total_gb = gpu.total_memory / 1024 ** 3
start_gb = torch.cuda.max_memory_reserved() / 1024 ** 3
print(f"{gpu.name} — {total_gb:.1f} GiB total")
print(f"{start_gb:.1f} GiB already reserved (the 4-bit model weights)")

Tesla T4 — 14.6 GiB total
12.8 GiB already reserved (the 4-bit model weights)


This is the run. On a demo it is short; on a real fine-tune it is the part where you go and do something else. Either way the loss column is the only thing to watch.

In [20]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 199998, 'pad_token_id': 200017}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 933 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 92,454,912 of 21,007,212,096 (0.44% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.074143
2,1.640558
3,1.056302
4,0.835288
5,1.365567
6,0.954527
7,0.943315
8,1.182417
9,1.302899
10,0.944612


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-30/tokenizer_config.json.


In [21]:
# @title 📊 What the run actually cost { display-mode: "form" }
peak_gb = torch.cuda.max_memory_reserved() / 1024 ** 3
runtime_s = trainer_stats.metrics["train_runtime"]
print(f"Runtime:        {runtime_s:.0f} s ({runtime_s / 60:.1f} min)")
print(f"Peak reserved:  {peak_gb:.1f} GiB of {total_gb:.1f} GiB ({peak_gb / total_gb:.0%})")
print(f"Training added: {peak_gb - start_gb:.1f} GiB on top of the loaded model")

Runtime:        886 s (14.8 min)
Peak reserved:  13.6 GiB of 14.6 GiB (93%)
Training added: 0.8 GiB on top of the loaded model


**What just happened?** You ran a supervised fine-tune of a 20B model on a consumer GPU, and the last cell tells you what it cost. The number worth staring at is the last line: the *training* overhead on top of the loaded weights. That gap is what LoRA, 8-bit Adam and gradient checkpointing bought — with full fine-tuning it would be tens of gigabytes of gradients and optimizer state, and there would be no run to report.

Your figures will differ with GPU, dataset and knobs, and the demo run's loss curve is too short to mean much on its own. Treat this section as proof the loop turns, not as evidence the model improved — that claim needs an eval set, which is Section 11's closing argument.


## 9. Inference on the Tuned Model

No reload is needed — the adapters are live on the model object in this process, so generation now runs the frozen base **plus** the update you just trained. Use the same `ask()` helper from Section 4 and compare against what the base model did there.

The dataset taught reasoning in non-English languages, so the system prompt asks for one. This is the observable behaviour change: same question, same model weights underneath, different thinking.

In [22]:
tuned_messages = [
    {
        "role": "system",
        "content": "reasoning language: French\n\nYou are a helpful assistant that can solve mathematical problems.",
    },
    {"role": "user", "content": "Solve x⁴ - 6x³ + 11x² - 6x = 0"},
]

_ = ask(tuned_messages, reasoning_effort="medium", max_new_tokens=512)

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-08-21

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

reasoning language: French

You are a helpful assistant that can solve mathematical problems.<|end|><|start|>user<|message|>Solve x⁴ - 6x³ + 11x² - 6x = 

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0<|end|><|start|>assistant<|channel|>analysis<|message|>We need to solve the polynomial equation. We can factor: x(x^3 - 6x² + 11x - 6) = 0. Indeed x⁴ - 6x³ + 11x² - 6x = 6 * [x^3 - 6x]. Wait. Let's factor: x⁴ - 6x³ + 11x² - 6x = 0. That equals? Let's check the polynomial: x^4 - 6x^3 + 11x^2 - 6x etc. Usually the factorization would be (x-1)^3 maybe? Wait, maybe we should check: x^4-6x³+11x²-6x = x( x^3 - 6x² + ... ) But 6x?? We can factor out x from the left: x(x³ - 6x² + 11x - 6x³). Actually I'd factor out x at the second step: x*(x^3-6x²+11x-6x). That gives: x(x³ - 6x² + 11x - 6) = 0? Wait... The typical approach is to factor completely. The equation is x⁴-6x³+11x²-6x = 0. Let's check the coefficients: 1, -6, 11, -6, ...? That suggests polynomial may be like (x-1)(x-2)(x-? maybe? Actually constant term -? But we only have 4 terms. We can try factoring by grouping: x(x³ - 6x² + 11x - 6)=0. Let x=0. Then x=0? Wait: we had 6? Wait, we have a mistake: the polynomial is not symmetrical? 

**What just happened?** The model answered through its adapters. Whatever you see, notice what the comparison with Section 4 is and is not: it is a demonstration that training changed the model's behaviour, and it is *not* evidence that the model got better. One prompt is an anecdote. Two prompts are two anecdotes.

⏭️ *The way to turn this into a claim you can defend is the one you already know: hold out a slice of the data before training, run both the base model and the tuned model over it, and score them with the same judges you built in the evaluation lessons. A fine-tune that "feels better" is a fine-tune you cannot ship.*

## 10. Saving: Adapters, Merges, and Exports

What you trained is a set of adapter matrices, and that is what `save_pretrained` writes — megabytes, not gigabytes, plus a config recording which base model they belong to. Save the tokenizer beside them every time: adapters trained against one chat template and loaded with another produce a model that is subtly, confusingly wrong.

In [23]:
model.save_pretrained("lora_adapters")      # adapter weights + config (small!)
tokenizer.save_pretrained("lora_adapters")  # always save the tokenizer alongside

# Or push both to the Hub — https://huggingface.co/settings/tokens for a token:
# model.push_to_hub("your-username/gpt-oss-20b-lora", token="hf_...")
# tokenizer.push_to_hub("your-username/gpt-oss-20b-lora", token="hf_...")

Unsloth: Restored added_tokens_decoder metadata in lora_adapters/tokenizer_config.json.


('lora_adapters/tokenizer_config.json',
 'lora_adapters/chat_template.jinja',
 'lora_adapters/tokenizer.json')

Reloading in a fresh runtime is one call: the adapter config names the base model, so the loader fetches it, re-quantizes to 4 bits, and attaches your adapters on top.

In [24]:
if False:  # flip to True in a fresh runtime
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="lora_adapters",  # the base model is recorded in the adapter config
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

Adapters are the right artefact while you iterate — swappable, tiny, stackable on one shared base. But most serving stacks want a single set of weights, and that is what **merging** produces: fold `B @ A` back into `W` and write out a normal model, with no adapter machinery at inference time and no per-token overhead.

The trade is honest: a merged model is a **full-size copy** of the base, one per fine-tune. Merge when you are ready to deploy, not while you are experimenting — and check your disk before you run any of these.

In [25]:
# Each of these writes a FULL copy of the model. Check free disk space first.

if False:  # 16-bit merge — the portable choice for vLLM and most serving stacks
    model.save_pretrained_merged("merged_16bit", tokenizer, save_method="merged_16bit")

if False:  # quantized merge — smaller on disk, gpt-oss's native 4-bit format
    model.save_pretrained_merged("merged_mxfp4", tokenizer, save_method="mxfp4")

if False:  # merge and push in one step
    model.push_to_hub_merged(
        "your-username/gpt-oss-20b-tuned",
        tokenizer,
        save_method="merged_16bit",
        token="hf_...",
    )

**What just happened?** You produced two different deliverables from one training run. The adapter folder is what you version, share and iterate on; the merged export is what you hand to a serving stack. Keeping both is normal — and keeping the adapters is what lets you re-merge later against an updated base model without retraining.

## 11. Fine-Tune, Prompt, or RAG?

You now have all three tools within reach, which makes it worth restating the rule that chooses between them — because "I know how to fine-tune" is a poor reason to fine-tune.

**Fine-tuning changes behaviour. RAG changes knowledge. Prompting changes both, cheaply, up to a point.**

- **Reach for prompting first, always.** It is free, it is reversible, and it is where most of the gap usually is. A run of prompt iterations costs an afternoon; a fine-tune costs a dataset.
- **Reach for RAG when the gap is knowledge** — the model does not know your documents, your prices, yesterday's incident. Fine-tuning is a poor way to install facts: they go stale, you cannot cite them, and updating one means training again.
- **Reach for fine-tuning when the gap is behaviour** — a house format the model keeps drifting out of, a tone, a reasoning procedure, a schema it must never break, or a small model you want to behave like a large one for a narrow task. That last case is the cost argument, and it is often the strongest one: a tuned small open model you host yourself can be dramatically cheaper per request than a frontier API, which is exactly why the local recipe in this notebook matters.

They compose rather than compete. The common production shape is a fine-tuned model that follows your format reliably, retrieving your documents at query time, behind a prompt that still does real work. The question is never "which one" but "which one next".

📎 *The fuller comparison — including prompt tuning, and the ethics and dataset-bias considerations that come with changing a model's weights — is in **Fine-Tuning 101**.*

⏭️ *And the discipline that makes any of it a decision rather than a preference: measure. Split a held-out set off your data before you train, score base versus tuned with the judges from the evaluation lessons, and only then decide whether the adapter ships.*

## 🔑 Key Takeaways

- **LoRA freezes the model and trains a low-rank update beside it.** A `d × d` update becomes `B @ A` with `r ≪ d`, so you train `2r / d` of each adapted matrix — the frozen base cannot be forgotten, and the artefact is megabytes rather than gigabytes.
- **QLoRA is the product of two savings, not one.** 4-bit frozen weights make the model *fit*; LoRA plus 8-bit optimizer state makes training it *affordable*. Either alone leaves a 20B model off a free T4.
- **The knobs are decisions, and the defaults are starting points.** `r` is capacity, `lora_alpha / r` is volume, `target_modules` is reach, gradient checkpointing is the memory-for-time trade. A 30-step demo and a two-epoch fine-tune want genuinely different settings — say which one you are running.
- **Mask the prompt out of the loss.** Training on the user's turn spends your gradient budget teaching the model to predict its own inputs, and the loss curve will not warn you. Print the labels and check.
- **Quality beats quantity in the dataset.** Fine-tuning has no mechanism for ignoring a bad example; a few hundred examples you would be happy to see imitated beat tens of thousands you have not read.
- **Adapters for iteration, merged weights for serving** — and keep the adapters either way, so you can re-merge onto a newer base without retraining.
- **Prompt, retrieve, then tune.** Fine-tuning fixes behaviour, not knowledge, and it is only a decision once you have measured it against the base model on held-out data.